# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through data loading and exploration for the [FAIR^2 Mass Spectrometry Extracellular Vesicles dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant](https://mlcommons.org/croissant/) schema, available at this URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
We will load the metadata and records from the Croissant dataset using `mlcroissant`. This gives us structured metadata and access to data via record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (FAIR^2 dataset)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display main description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's explore the record sets, their `@id`s, and the fields/columns present. All references are made explicitly using the `@id` field for full traceability.

In [ ]:
# Examine the available Record Sets and their @ids
record_sets = []
for rs in metadata.record_sets:
    print(f"RecordSet: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, dataType: {f.data_type})")
    record_sets.append(rs.id)

if not record_sets:
    print("No record sets found in metadata. Please refer to the FAIR^2 dataset documentation for details.")

# Preview some records from the main record set (if available)
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"\nFirst 2 records from RecordSet '{main_record_set_id}':")
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        if i >= 2:
            break
        print(rec)

## 3. Data Extraction
Load data from each record set you wish to analyze into a pandas DataFrame. Reference record sets and fields by their `@id`s for reproducibility.

We'll preview the columns and a few records for the primary record set.

In [ ]:
# Extract data from each record set into dataframes
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet: {record_set_id} | #rows: {len(df)} | Columns: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"RecordSet: {record_set_id} has no records.")

## 4. Exploratory Data Analysis (EDA)
Now let's process the main data. We'll select a numeric field by its `@id` for analysis (e.g., peptide or intensity-related fields), apply filtering, normalization, and grouping, all using the field `@id`s found above.

In [ ]:
import numpy as np

# Choose the first record set as main (adjust if you have a specific @id)
main_record_set_id = record_sets[0] if record_sets else None
df = dataframes[main_record_set_id] if main_record_set_id else pd.DataFrame()

if not df.empty:
    # List all available columns
    print("Columns:", df.columns.tolist())

    # Pick example numeric field @id -- modify based on actual fields (by inspecting above)
    # Example guesses: 'cr:Peptide_Count', 'cr:Molecular_Weight', 'cr:Normalized_Abundance'
    numeric_field_id = None
    for col in df.columns:
        if (('abundance' in col.lower()) or ('count' in col.lower()) or ('mw' in col.lower())):
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        print(f"Using numeric field for EDA: '{numeric_field_id}'")

        # Convert to float if needed and drop NaN
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        filtered_df = df[df[numeric_field_id] > 0].copy()

        # Remove high-valued outliers (simple 99% quantile clip)
        upper = filtered_df[numeric_field_id].quantile(0.99)
        filtered_df = filtered_df[filtered_df[numeric_field_id] < upper]
        print(f"Filtered records where 0 < {numeric_field_id} < {upper:.2f}")

        # Normalize
        mean_val = filtered_df[numeric_field_id].mean()
        std_val = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val

        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If there appears to be a label/category field, try grouping
        potential_group_fields = [c for c in df.columns if ('sample' in c.lower()) or ('group' in c.lower()) or ('label' in c.lower())]
        group_field = potential_group_fields[0] if potential_group_fields else None
        if group_field is not None:
            print(f"\nGrouping by field: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(grouped.head())
    else:
        print("No obvious numeric field found—please inspect available columns above.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize distributions and relationships in the data using the selected numeric fields. You might want to examine histograms, boxplots, or scatter relationships if there are multiple numeric fields.

In [ ]:
import matplotlib.pyplot as plt

if not df.empty and numeric_field_id:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    filtered_df[numeric_field_id].hist(ax=ax[0], bins=30)
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    ax[0].set_ylabel('Frequency')

    filtered_df.boxplot(column=numeric_field_id, ax=ax[1])
    ax[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    if group_field:
        # Show group means (bar plot)
        group_means = filtered_df.groupby(group_field)[numeric_field_id].mean().sort_values()
        group_means.plot(kind='bar', figsize=(10,4), title=f"Mean {numeric_field_id} by {group_field}")
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Load data and choose a numeric_field_id above to visualize.")

## 6. Conclusion
- We loaded and explored record sets and fields from the FAIR^2 extracellular vesicles dataset using the `mlcroissant` library.
- Using field and record set `@id`s ensures reproducible data processing across Croissant-compliant datasets.
- Basic EDA and visualization allowed for initial inspection of protein-related numeric fields; further domain-specific statistical or downstream ML tasks can now be pursued.

For more details, see the [FAIR^2 data documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) or refer to the Croissant metadata objects in code for deeper field-level metadata.